In [ ]:
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from ibm_watsonx_ai import Credentials
from langchain_ibm import WatsonxLLM, WatsonxEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from huggingface_hub import HfFolder

In [ ]:
import gradio as gr

In [ ]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [ ]:
## LLM   # Initialize the LLM
def get_llm():
   # model_id = 'mistralai/mixtral-8x7b-base'
    model_id = 'ibm/granite-3-2-8b-instruct'
    #model_id = 'meta-llama/llama-3-3-70b-instruct'

    parameters = {
        GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
        GenParams.TEMPERATURE: 0.5, # this randomness or creativity of the model's responses
    }

    project_id = "skills-network"

    watsonx_llm = WatsonxLLM(
        model_id=model_id,
        url="https://us-south.ml.cloud.ibm.com",
        project_id=project_id,
        params=parameters,
    )
    return watsonx_llm


In [ ]:
# Task1. Load document using LangChain for different sources
## Document loader   #set up the PDF document loader
def document_loader(file):
    loader = PyPDFLoader(file)
    loaded_document = loader.load()
    return loaded_document


In [ ]:
# Task2:Splitting long documents using text splitters
## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks


In [ ]:
# Task 3: Embed documents
## Embedding model
def watsonx_embedding():
    embed_params = {
        EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
        EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
    }
    watsonx_embedding = WatsonxEmbeddings(
        model_id="ibm/slate-125m-english-rtrvr-v2",
        url="https://us-south.ml.cloud.ibm.com",
        project_id="skills-network",
        params=embed_params,
    )
    return watsonx_embedding


In [ ]:
# Task 4: Create and configure vector databases to store embeddings
## Vector db
def vector_database(chunks):
    ids = [str(i) for i in range(0, len(chunks))]
    embedding_model = watsonx_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model, ids=ids)
    return vectordb


In [ ]:
# Task 5: Develop a retriever to fetch document segments based on queries
## Retriever
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever(search_kwargs={"k": 1})
    return retriever



In [ ]:
## QA Chain
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)
    qa = RetrievalQA.from_chain_type(llm=llm, 
                                    chain_type="stuff", 
                                    retriever=retriever_obj, 
                                    return_source_documents=True)
    response = qa.invoke(query)
    return response['result']


In [ ]:
# Create Gradio interface
rag_application = gr.Interface(
    fn=retriever_qa,
    allow_flagging="never",
    inputs=[
        gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),  # Drag and drop file upload
        gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
    ],
    outputs=gr.Textbox(label="Here you see answers to your queries"),
    title="Question-Answering BOT",
    description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
)


In [ ]:
# Launch the app
rag_application.launch(server_name="127.0.0.1", server_port= 7860)